# Check Citation Faithfulness in Cohere RAG

Cohere's Chat API grounds answers in the `documents` you pass and returns
**citations** — spans of the reply, each linked to the source document(s) it came
from. That is exactly the metadata you need to catch a citation *before* it
reaches a user. This notebook adds a small, deterministic faithfulness check on
top of Cohere citations.

The failure modes worth catching are the ones that read as authoritative:

- **fabricated** — a cited span that appears in none of the source documents;
- **frankenquote** — every word is real, but the exact span was never written
  contiguously in the source;
- **misattributed** — a real span, but the citation points at the wrong document.

A judge model asked "does this support the claim?" waves all three through — they
look fluent and supportive. So we ask the cheaper, prior question first, with no
model and no tokens: **does the cited text appear verbatim in the document it is
attributed to?**

This is the standalone, framework-agnostic gate from
[`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate),
inlined here so the notebook has no extra dependency.

In [ ]:
import re


def normalize(text: str) -> str:
    """Case/typography/whitespace-insensitive form for verbatim matching."""
    text = text.lower()
    text = re.sub(r"[‘’]", "'", text)
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[–—]", "-", text)
    text = re.sub(r"[^a-z0-9%.]+", " ", text)
    return " ".join(text.split())


def gate(cited_text: str, cited_doc_id: str, docs: dict) -> str:
    """Return 'found' | 'misattributed' | 'not_found'. Fails closed on empty text."""
    q = normalize(cited_text)
    if not q:
        return "not_found"
    cited = docs.get(cited_doc_id)
    if cited is not None and q in normalize(cited):
        return "found"
    if any(q in normalize(t) for d, t in docs.items() if d != cited_doc_id):
        return "misattributed"
    return "not_found"

## 1. Run it offline on Cohere-shaped citations

So the notebook is runnable in CI without an API key, here is a small knowledge
base and a set of citations in the shape Cohere's Chat API returns them
(`start`, `end`, `text`, and the `sources` they are attributed to). One citation
is faithful; the others are the three planted failure modes.

In [ ]:
# id -> document text (as you would pass to co.chat(documents=...))
DOCS = {
    "doc_0": "The Aptera solar EV has a claimed range of 400 miles on a single charge.",
    "doc_1": "Its roof-mounted solar array adds up to 40 miles of range per day in ideal sun.",
}

# Citations in the shape Cohere returns (text = the exact span, sources = doc ids).
CITATIONS = [
    {"text": "range of 400 miles on a single charge", "doc_id": "doc_0",   # faithful
     "claim": "The car goes 400 miles per charge."},
    {"text": "adds up to 40 miles of range per day", "doc_id": "doc_0",    # real span, wrong doc
     "claim": "Solar adds 40 miles/day."},
    {"text": "400 miles of range per day from solar", "doc_id": "doc_1",   # frankenquote
     "claim": "Solar alone gives 400 miles/day."},
    {"text": "a top speed of 110 miles per hour", "doc_id": "doc_0",       # fabricated
     "claim": "Top speed is 110 mph."},
]

for c in CITATIONS:
    status = gate(c["text"], c["doc_id"], DOCS)
    faithful = "OK  " if status == "found" else "FLAG"
    print(f"{faithful} [{status:>13}]  {c['claim']}")

`found` citations are safe to surface; `misattributed`, `not_found` (fabricated
or frankenquote) should be flagged or dropped before the answer reaches a user —
all decided deterministically, for zero tokens.

## 2. Wire it to the live Cohere Chat API

With an API key, ground a real answer in documents, then run the same gate over
the citations Cohere returns. This cell needs `COHERE_API_KEY` and is not run in
CI.

In [ ]:
# pip install cohere
import os

if not os.getenv("COHERE_API_KEY"):
    print("Set COHERE_API_KEY to run the live example.")
else:
    import cohere

    co = cohere.ClientV2()
    documents = [
        {"id": "doc_0", "data": {"text": DOCS["doc_0"]}},
        {"id": "doc_1", "data": {"text": DOCS["doc_1"]}},
    ]
    resp = co.chat(
        model="command-r-plus",
        messages=[{"role": "user", "content": "What is the Aptera's range, and how much does solar add per day?"}],
        documents=documents,
    )

    # Build the id -> text map from the same documents we grounded on.
    doc_text = {d["id"]: d["data"]["text"] for d in documents}

    for cit in (resp.message.citations or []):
        span = cit.text
        for src in cit.sources:
            # ChatV2 source ids look like "doc_0", matching the document ids above.
            doc_id = getattr(src, "document", {}).get("id") if hasattr(src, "document") else src.id
            status = gate(span, doc_id, doc_text)
            flag = "OK  " if status == "found" else "FLAG"
            print(f"{flag} [{status:>13}]  {span!r} -> {doc_id}")

## Notes

- Cohere cites **spans of the generated reply**, so the gate's "does this text
  exist verbatim in the cited document?" question maps directly onto
  `citation.text` vs the document it points at — no offset bookkeeping needed.
- The gate is the cheap first stage. For the harder, genuinely ambiguous case —
  a real, correctly-attributed span that may still not *support* the claim at full
  strength — pair it with the burden-of-proof LLM judge in
  [`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate),
  which fails closed on unparseable output.